In [1]:
# Environment setup
import os
import sys

IS_COLAB = "google.colab" in sys.modules
IS_CODESPACES = os.path.exists("/.devcontainer") or os.path.exists("/workspaces")
IS_LOCAL = not (IS_COLAB or IS_CODESPACES)
print(f"🚀 Environment: {'Colab' if IS_COLAB else 'Codespaces' if IS_CODESPACES else 'Local'}")

while not os.path.exists('data') and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir('..')
print(f"📁 Working directory: {os.getcwd()}")

print("✅ Environment ready!")

# 07 - Analyst Sentiment Features

**Goal**: Add analyst recommendations and price targets to quarterly panel.

**Source**: yfinance (ticker.recommendations, ticker.analyst_price_targets)

**Output**: Add 7 columns to quarterly_panel.csv

In [2]:
import pandas as pd
import yfinance as yf
import warnings
warnings.filterwarnings('ignore')

# Current tickers
TICKERS = ['AMT', 'AMZN', 'CCI', 'DLR', 'EQIX', 'GOOGL', 'IRM', 'META', 'MSFT', 'NVDA', 'SBAC']

print(f"✅ Libraries imported for {len(TICKERS)} tickers")

## Step 1: Fetch Analyst Recommendations

In [3]:
# Fetch recommendations for each ticker
recommendations = []

for ticker in TICKERS:
    try:
        t = yf.Ticker(ticker)
        recs = t.recommendations
        if recs is not None and not recs.empty:
            recs = recs.reset_index()
            recs['ticker'] = ticker
            recommendations.append(recs)
    except Exception as e:
        print(f"⚠️ {ticker}: {e}")

if recommendations:
    df_recs = pd.concat(recommendations, ignore_index=True)
    # Take the most recent period row
    df_recs = df_recs.sort_values('period').groupby('ticker').last().reset_index()
    df_recs = df_recs[['ticker', 'strongBuy', 'buy', 'hold', 'sell', 'strongSell']]
    print(f"✅ Recommendations: {len(df_recs)} tickers")
    print(df_recs.head())
else:
    print("⚠️ No recommendations fetched")

## Step 2: Fetch Analyst Price Targets

In [4]:
# Fetch analyst price targets
price_targets = []

for ticker in TICKERS:
    try:
        t = yf.Ticker(ticker)
        targets = t.analyst_price_targets
        if targets and isinstance(targets, dict):
            targets['ticker'] = ticker
            price_targets.append(targets)
    except Exception as e:
        print(f"⚠️ {ticker}: {e}")

if price_targets:
    df_targets = pd.DataFrame(price_targets)
    df_targets = df_targets[['ticker', 'current', 'high', 'low', 'mean', 'median']]
    df_targets = df_targets.rename(columns={
        'current': 'price_current',
        'high': 'target_high',
        'low': 'target_low',
        'mean': 'target_mean',
        'median': 'target_median'
    })
    # Calculate premium
    df_targets['target_premium_pct'] = ((df_targets['target_mean'] - df_targets['price_current']) / df_targets['price_current'] * 100).round(2)
    print(f"✅ Price targets: {len(df_targets)} tickers")
    print(df_targets.head())
else:
    print("⚠️ No price targets fetched")

## Step 3: Merge into quarterly_panel

In [5]:
# Load current quarterly panel
df_panel = pd.read_csv('data/processed/quarterly_panel.csv')
print(f"📊 Quarterly panel: {len(df_panel)} rows × {len(df_panel.columns)} cols")

# Merge recommendations
df_panel = df_panel.merge(df_recs, on='ticker', how='left')
print(f"✅ After recommendations: {len(df_panel)} rows × {len(df_panel.columns)} cols")

# Merge price targets
df_panel = df_panel.merge(df_targets, on='ticker', how='left')
print(f"✅ After price targets: {len(df_panel)} rows × {len(df_panel.columns)} cols")

# Fill missing with median per ticker
for col in ['strongBuy', 'buy', 'hold', 'sell', 'strongSell', 'target_premium_pct']:
    if col in df_panel.columns:
        df_panel[col] = df_panel.groupby('ticker')[col].transform(lambda x: x.fillna(x.median()))
        df_panel[col] = df_panel[col].fillna(0)

print("✅ Missing values filled")

## Step 4: Save

In [6]:
# Save
output_path = 'data/processed/quarterly_panel.csv'
df_panel.to_csv(output_path, index=False)
print(f"✅ Saved: {output_path}")
print(f"   Rows: {len(df_panel):,}, Columns: {len(df_panel.columns)}")
print(f"\n📊 New columns: {[c for c in df_panel.columns if c in ['strongBuy', 'buy', 'hold', 'sell', 'strongSell', 'price_current', 'target_high', 'target_low', 'target_mean', 'target_median', 'target_premium_pct']]}")

## Summary

✅ **Analyst Sentiment Added**

- 7 new columns from yfinance
- Recommendations: strongBuy, buy, hold, sell, strongSell
- Price targets: current, high, low, mean, median, premium%
- All 11 tickers covered